# Automated Growth-Rate Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.signal import savgol_filter
from scipy.stats import linregress

import seaborn as sns

## Load experiment

In [ ]:
file = "../examples/example_log.csv"

df = pd.read_csv(file)

df["timestamp"] = pd.to_datetime(df["timestamp"])

df.head()

## Convert time axis

In [ ]:
df["time_s"] = (df["timestamp"] - df["timestamp"].iloc[0]).dt.total_seconds()

time = df["time_s"].values
interface = df["filtered_interface_mm"].values

## Savitzky-Golay-Smoothing

Suppress measurement noise.

In [ ]:
smoothed_interface = savgol_filter(interface, 101, 3)

plt.figure(figsize=(10,5))

plt.plot(time, interface, alpha=0.3, label="raw")
plt.plot(time, smoothed_interface, label="smoothed")

plt.xlabel("time (s)")
plt.ylabel("interface position (mm)")
plt.legend()

## Instantaneous Growth Rate

Velocity is calculated from the derivative.

In [ ]:
velocity = np.gradient(smoothed_interface, time)

growth_rate = velocity * 60

plt.figure(figsize=(10,5))

plt.plot(time, growth_rate)

plt.xlabel("time (s)")
plt.ylabel("growth rate (mm/min)")
plt.title("Instantaneous Growth Rate")

## Moving Window Regression

In [ ]:
window = 600

rates = []
times = []

for i in range(len(time)-window):

    t = time[i:i+window]
    h = interface[i:i+window]

    slope,_,_,_,_ = linregress(t,h)

    rates.append(slope*60)
    times.append(t.mean())

plt.figure(figsize=(10,5))

plt.plot(times, rates)

plt.xlabel("time (s)")
plt.ylabel("growth rate (mm/min)")
plt.title("Moving Window Growth Rate")

## Growth Stability Metric

Typical interpretation:
- <0.05 very stable growth
- 0.05–0.15 moderate fluctuations
- \>0.15 unstable interface

In [ ]:
rate_std = np.std(rates)
rate_mean = np.mean(rates)

stability = rate_std / rate_mean

print("Growth stability coefficient:", stability)

## Interface Acceleration

Detect growth instabilities.

In [ ]:
acceleration = np.gradient(velocity, time)

plt.figure(figsize=(10,5))

plt.plot(time, acceleration)

plt.xlabel("time (s)")
plt.ylabel("interface acceleration (mm/s²)")

## Temperature vs Growth Correlation

In [ ]:
plt.figure(figsize=(7,6))

sns.scatterplot(
    x=df["temperature_C"],
    y=df["growth_rate_mm_min"]
)

plt.xlabel("Temperature (°C)")
plt.ylabel("Growth rate (mm/min)")

## Automatic Experiment Summary

In [ ]:
print("Experiment duration (hours):",
      time.max()/3600)

print("Mean growth rate (mm/min):",
      np.mean(growth_rate))

print("Growth stability:",
      stability)

print("Maximum growth rate:",
      np.max(growth_rate))

print("Minimum growth rate:",
      np.min(growth_rate))